In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

print("Libraries imported successfully!")

Libraries imported successfully!


In [15]:
# Mã cổ phiếu và chỉ số thị trường
stock_ticker = "XOM"
market_ticker = "^GSPC"
# Khoảng thời gian nghiên cứu
start_date = "2024-01-01"
end_date = "2026-09-30"

print("Stock:", stock_ticker)
print("Market:", market_ticker)
print("Period:", start_date, "to", end_date)

Stock: XOM
Market: ^GSPC
Period: 2024-01-01 to 2026-09-30


In [16]:
# Download dữ liệu giá
data = yf.download(
    [stock_ticker, market_ticker],
    start=start_date,
    end=end_date,
    auto_adjust=False
)
data.head()

[*********************100%***********************]  2 of 2 completed


Price       Adj Close                    Close                     High  \
Ticker            XOM        ^GSPC         XOM        ^GSPC         XOM   
Date                                                                      
2024-01-02  93.601608  4742.830078  102.360001  4742.830078  103.099998   
2024-01-03  94.388016  4704.810059  103.220001  4704.810059  103.620003   
2024-01-04  93.565025  4688.680176  102.320000  4688.680176  104.570000   
2024-01-05  93.848488  4697.240234  102.629997  4697.240234  103.400002   
2024-01-08  92.284813  4763.540039  100.919998  4763.540039  101.040001   

Price                           Low                     Open               \
Ticker            ^GSPC         XOM        ^GSPC         XOM        ^GSPC   
Date                                                                        
2024-01-02  4754.330078  100.849998  4722.669922  100.919998  4745.200195   
2024-01-03  4729.290039  101.660004  4699.709961  102.269997  4725.069824   
2024-01-04  4726.779785  102.050003  4687.529785  104.080002  4697.419922   
2024-01-05  4721.490234  102.129997  4682.109863  103.169998  4690.569824   
2024-01-08  4764.540039   98.900002  4699.819824  100.730003  4703.700195   

Price         Volume              
Ticker           XOM       ^GSPC  
Date                              
2024-01-02  23483000  3743050000  
2024-01-03  23490800  3950760000  
2024-01-04  19395200  3715480000  
2024-01-05  15827400  3844370000  
2024-01-08  23370100  3742320000

In [17]:
# Lấy Adjusted Close
adj_close = data["Adj Close"].copy()
# Đổi tên cột cho dễ sử dụng
adj_close.columns = ["XOM", "SP500"]

adj_close.head()

,XOM,SP500
Date,,
2024-01-02,93.601608,4742.830078
2024-01-03,94.388016,4704.810059
2024-01-04,93.565025,4688.680176
2024-01-05,93.848488,4697.240234
2024-01-08,92.284813,4763.540039


In [18]:
print("Dataset shape:", adj_close.shape)
print("\nColumn names:")
print(adj_close.columns.tolist())
print("\nData types:")
print(adj_close.dtypes)
print("\nMissing values:")
print(adj_close.isnull().sum())
print("\nDuplicate dates:")
print(adj_close.index.duplicated().sum())

Dataset shape: (687, 2)

Column names:
['XOM', 'SP500']

Data types:
XOM      float64
SP500    float64
dtype: object

Missing values:
XOM      0
SP500    0
dtype: int64

Duplicate dates:
0


In [19]:
adj_close.describe()

,XOM,SP500
count,687.000000,687.000000
mean,117.911421,6202.554701
std,20.523223,821.749319
min,88.517357,4688.680176
25%,104.375126,5554.995117
50%,109.690575,6049.240234
75%,130.997124,6844.360107
max,169.320007,7798.990234


In [20]:
# Xóa các dòng bị thiếu dữ liệu
clean_data = adj_close.dropna().copy()
# Xóa các ngày bị trùng nếu có
clean_data = clean_data[~clean_data.index.duplicated(keep="first")]
# Sắp xếp theo thời gian
clean_data = clean_data.sort_index()

print("Shape after cleaning:", clean_data.shape)

print("\nMissing values after cleaning:")
print(clean_data.isnull().sum())

clean_data.head()

Shape after cleaning: (687, 2)

Missing values after cleaning:
XOM      0
SP500    0
dtype: int64


,XOM,SP500
Date,,
2024-01-02,93.601608,4742.830078
2024-01-03,94.388016,4704.810059
2024-01-04,93.565025,4688.680176
2024-01-05,93.848488,4697.240234
2024-01-08,92.284813,4763.540039


In [21]:
print("First trading day:", clean_data.index.min())
print("Last trading day:", clean_data.index.max())
print("Number of observations:", len(clean_data))

First trading day: 2024-01-02 00:00:00
Last trading day: 2026-09-28 00:00:00
Number of observations: 687


In [22]:
# Tính log-return hàng ngày
returns = np.log(clean_data / clean_data.shift(1))
# Đổi tên biến
returns.columns = ["XOM_Return", "SP500_Return"]
# Xóa dòng đầu tiên vì không có giá ngày trước đó
returns = returns.dropna()

returns.head()

,XOM_Return,SP500_Return
Date,,
2024-01-03,0.008367,-0.008049
2024-01-04,-0.008757,-0.003434
2024-01-05,0.003025,0.001824
2024-01-08,-0.016802,0.014016
2024-01-09,-0.012463,-0.001479


In [23]:
print("Return dataset shape:", returns.shape)
print("\nMissing values:")
print(returns.isnull().sum())
print("\nSummary statistics:")
print(returns.describe())

Return dataset shape: (686, 2)

Missing values:
XOM_Return      0
SP500_Return    0
dtype: int64

Summary statistics:
       XOM_Return  SP500_Return
count  686.000000    686.000000
mean     0.000804      0.000703
std      0.014836      0.009594
min     -0.074676     -0.061609
25%     -0.007506     -0.003494
50%      0.000979      0.000944
75%      0.010067      0.005772
max      0.048710      0.090895


In [24]:
final_data = clean_data.join(returns, how="inner")
final_data.head(10)

,XOM,SP500,XOM_Return,SP500_Return
Date,,,,
2024-01-03,94.388016,4704.810059,0.008367,-0.008049
2024-01-04,93.565025,4688.680176,-0.008757,-0.003434
2024-01-05,93.848488,4697.240234,0.003025,0.001824
2024-01-08,92.284813,4763.540039,-0.016802,0.014016
2024-01-09,91.141777,4756.500000,-0.012463,-0.001479
2024-01-10,90.245621,4783.450195,-0.009881,0.005650
2024-01-11,90.227341,4780.240234,-0.000203,-0.000671
2024-01-12,91.397804,4783.830078,0.012889,0.000751
2024-01-16,89.331200,4765.979980,-0.022871,-0.003738


In [25]:
final_data.to_csv("../data/XOM_SP500_2024_2026.csv")
print("Processed data saved successfully!")

Processed data saved successfully!
